# 02 — Exploración de BBC News Classification Dataset e IMDB 50K Reviews

**BBC News** — Cita: Greene, D., & Cunningham, P. (2006). *Practical Solutions to the Problem of Diagonal Dominance in Kernel Document Clustering.* ICML 2006. Fuente: http://mlg.ucd.ie/datasets/bbc.html (tambien en Kaggle: `learn-ai-bbc`).

**IMDB 50K Reviews** — Cita: Maas, A. L., et al. (2011). *Learning Word Vectors for Sentiment Analysis.* ACL-HLT 2011. Fuente: https://ai.stanford.edu/~amaas/data/sentiment/ (tambien en Kaggle: `lakshmi25npathi/imdb-dataset-of-50k-movie-reviews`).

Este cuaderno es el **plan de trabajo** para reemplazar `RAW_BBC_NEWS` y `RAW_IMDB_REVIEWS` (en `js/embeddings-data.js`) por documentos y vectores reales.

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "../data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

BBC_CSV_PATH = os.path.join(DATA_DIR, "bbc-text.csv")       # columnas: category, text
IMDB_CSV_PATH = os.path.join(DATA_DIR, "IMDB_Dataset.csv")   # columnas: review, sentiment

print("BBC News fuente:", "http://mlg.ucd.ie/datasets/bbc.html")
print("IMDB fuente:", "https://ai.stanford.edu/~amaas/data/sentiment/")

## Paso 1 — Descarga (pendiente de autorización)

BBC y IMDB no tienen un endpoint único de descarga directa sin pasar por Kaggle (requiere API key) o por el `.tar.gz` del dataset de Maas et al. para IMDB. Esta celda queda documentada para ejecutarse en la siguiente fase, con las credenciales/API del usuario si usa Kaggle, o el tarball directo de Stanford para IMDB.

In [ ]:
# Opcion A: Kaggle API (requiere kaggle.json configurado)
# !kaggle datasets download -d yufengdev/bbc-fulltext-and-category -p ../data/raw --unzip
# !kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews -p ../data/raw --unzip

# Opcion B: IMDB tarball oficial de Stanford (sin Kaggle)
# import urllib.request, tarfile
# url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
# urllib.request.urlretrieve(url, os.path.join(DATA_DIR, "aclImdb_v1.tar.gz"))
# with tarfile.open(os.path.join(DATA_DIR, "aclImdb_v1.tar.gz")) as t:
#     t.extractall(DATA_DIR)
print("Descarga pendiente de autorizacion explicita del usuario.")

## Paso 2 — Vectorización densa a 50D (sustituto real de los vectores sintéticos)

Se usará TF-IDF + reducción SVD (`TruncatedSVD`) a 50 dimensiones por documento, el equivalente real y reproducible a los `vector(50)` que hoy son sintéticos en `embeddings-data.js`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

def documents_to_dense_vectors(texts, dims=50, random_state=42):
    """Convierte una lista de textos reales en vectores densos normalizados (norma L2 = 1),
    compatibles con el contrato que valida validate-dataset.js.
    """
    tfidf = TfidfVectorizer(max_features=20000, stop_words=None)
    X = tfidf.fit_transform(texts)
    svd = TruncatedSVD(n_components=dims, random_state=random_state)
    dense = svd.fit_transform(X)
    dense = normalize(dense, norm="l2")
    return dense, svd.explained_variance_ratio_

# Ejemplo de uso (tras cargar los CSV reales):
# bbc_df = pd.read_csv(BBC_CSV_PATH)
# vectors, var_ratio = documents_to_dense_vectors(bbc_df['text'].tolist())
# print('Varianza explicada (50D):', var_ratio.sum())
print("Listo para ejecutar tras la descarga de datos reales.")

## Paso 3 — Muestreo balanceado y exportación a JSON

Para mantener el proyector 3D performante en el navegador, se seleccionará una muestra balanceada (p. ej. ~35-40 artículos BBC repartidos entre 5 categorías, y ~16-20 reseñas IMDB balanceadas pos/neg), igual al tamaño que hoy manejan los datos sintéticos, pero con contenido y vectores reales.

In [ ]:
# Ejemplo (tras vectorizar):
# sample = bbc_df.groupby('category', group_keys=False).apply(lambda g: g.sample(7, random_state=42))
# records = [
#     {"title": row.text[:60], "category": row.category, "snippet": row.text[:220], "vector": vec.tolist()}
#     for row, vec in zip(sample.itertuples(), vectors[sample.index])
# ]
# pd.DataFrame(records).to_json("../data/processed/bbc_news_real.json", orient="records", force_ascii=False)
print("Exportacion pendiente de la fase de descarga/integracion.")

## Paso 4 — Verificación de vecinos más cercanos (K-NN) con datos reales

Se reutilizará `vector-ops.js knn` (ya implementado) sobre los vectores reales exportados, verificando que los artículos más similares pertenezcan a la misma categoría temática — igual que hoy, pero con una similitud coseno realista (no artificialmente cercana a 1.0).